## With Complete datasets

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from transformers import RobertaTokenizerFast, RobertaForTokenClassification
from torch.optim import AdamW
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pickle
def load_data(file_path):
    df = pd.read_csv(file_path)
    sentences = df['Sentence'].tolist()
    labels = df['Labels'].apply(lambda x: x.split()).tolist()
    return sentences, labels

class EntityDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_len, label_encoder):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label_encoder = label_encoder

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, item):
        sentence = str(self.sentences[item])
        label = self.labels[item]

        words = sentence.split()
        word_labels = self.label_encoder.transform(label)

        word_labels = word_labels[:len(words)]
        word_labels = np.pad(word_labels, (0, len(words) - len(word_labels)), 'constant', constant_values=0)

        tokenized_inputs = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
        )

        word_ids = tokenized_inputs.word_ids()

        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_ids.append(word_labels[word_idx])

        return {
            'input_ids': torch.tensor(tokenized_inputs['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(tokenized_inputs['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(label_ids, dtype=torch.long)
        }

def prepare_data(file_paths, tokenizer, max_len, test_size=0.2, random_state=42):
    all_sentences = []
    all_labels = []
    unique_labels = set()

    # Load all files and collect sentences, labels, and unique labels across all files
    for file_path in file_paths:
        sentences, labels = load_data(file_path)
        all_sentences.extend(sentences)
        all_labels.extend(labels)
        
        # Update the set of unique labels to include any new labels from this file
        unique_labels.update(label for sublist in labels for label in sublist)

    # Initialize and fit the label encoder with all unique labels
    label_encoder = LabelEncoder()
    label_encoder.fit(list(unique_labels))

    # Split the data into training and validation sets
    train_sentences, val_sentences, train_labels, val_labels = train_test_split(
        all_sentences, all_labels, test_size=test_size, random_state=random_state
    )

    # Create datasets with the fitted label encoder
    train_dataset = EntityDataset(train_sentences, train_labels, tokenizer, max_len, label_encoder)
    val_dataset = EntityDataset(val_sentences, val_labels, tokenizer, max_len, label_encoder)
    print(unique_labels,"****")
    return train_dataset, val_dataset, label_encoder


def train_model(model, train_loader, val_loader, optimizer, num_epochs, device):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                val_loss += outputs.loss.item()

        print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {total_loss/len(train_loader):.4f}, Validation Loss: {val_loss/len(val_loader):.4f}")

    return model

# Include 'ner_skill_data.csv' in file_paths
file_paths = ['ner_name_data.csv', 'ner_address_data.csv', 'ner_skill_data.csv','ner_education_data1.csv','ner_experience_data.csv']

# Existing main function
def main(file_paths, max_len=128, batch_size=16, num_epochs=5, learning_rate=2e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base', add_prefix_space=True)

    # Prepare data (now with the new file added)
    train_dataset, val_dataset, label_encoder = prepare_data(file_paths, tokenizer, max_len)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize the model (adapt to new number of labels)
    model = RobertaForTokenClassification.from_pretrained('roberta-base', num_labels=len(label_encoder.classes_))
    model.to(device)

    # Set up the optimizer
    optimizer = AdamW(model.parameters(), lr=learning_rate)

    # Train the model
    trained_model = train_model(model, train_loader, val_loader, optimizer, num_epochs, device)
    
    # Save the model, tokenizer, and label encoder
     # Save the model, tokenizer, and label encoder as .pkl
    save_path = "final_new_model_combined_full.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump({
            'model_state_dict': trained_model.state_dict(),
            'tokenizer': tokenizer,
            'label_encoder': label_encoder
        }, f)
    print(f"Model, tokenizer, and label encoder saved to {save_path}")

# Usage
main(file_paths)  

c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch
from transformers import RobertaTokenizerFast, RobertaForTokenClassification
import numpy as np; 
import pickle
# Function to load the saved model, tokenizer, and label encoder

def load_model(model_path):
    with open(model_path, 'rb') as f:
        checkpoint = pickle.load(f)
    model = RobertaForTokenClassification.from_pretrained('roberta-base', num_labels=len(checkpoint['label_encoder'].classes_))
    model.load_state_dict(checkpoint['model_state_dict'])
    tokenizer = checkpoint['tokenizer']
    label_encoder = checkpoint['label_encoder']
    model.eval()  # Set model to evaluation mode
    return model, tokenizer, label_encoder


def predict(sentence, model, tokenizer, label_encoder, max_len=128, device='cpu'):
    # Tokenize the sentence
    tokenized_inputs = tokenizer(
        sentence.split(),
        is_split_into_words=True,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_attention_mask=True
    )

    input_ids = tokenized_inputs['input_ids'].to(device)
    attention_mask = tokenized_inputs['attention_mask'].to(device)

    # Get predictions from the model
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=2).cpu().numpy()

    # Map predictions back to labels
    word_ids = tokenized_inputs.word_ids()
    label_ids = predictions[0]
    labels = []
    
    # Only include the first token of a word and exclude subword parts
    prev_word_idx = None
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != prev_word_idx:  # Only predict once per word
            labels.append(label_encoder.inverse_transform([label_ids[idx]])[0])
        prev_word_idx = word_idx

    # Get tokens
    tokens = sentence.split()

    return list(zip(tokens, labels))

# Load the saved model
model_path = 'final_new_model_combined_full.pkl'
model, tokenizer, label_encoder = load_model(model_path)



Exception: Error while attempting to unpickle Tokenizer: data did not match any variant of untagged enum ModelWrapper at line 1 column 1556138

In [45]:
# Function to continuously prompt for input sentences
def Prediction(model, tokenizer, label_encoder, device):
    model.to(device)  # Ensure the model is on the correct device
    while True:
        # Example usage of prediction
        sentence = input("Enter a sentence for NER (type 'exit' or 'quit' to stop): ")
        if sentence.lower() in ['exit', 'quit']:
            print("Exiting the NER prediction program.")
            break
        
        # Get predictions
        predictions = predict(sentence, model, tokenizer, label_encoder, device=device)
        for token, label in predictions:
            print(f"Token: {token}, Label: {label}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Start the continuous prediction
Prediction(model, tokenizer, label_encoder, device)

Token: I, Label: O
Token: done, Label: O
Token: ph.D,, Label: B-Degree
Token: Master, Label: I-Degree
Token: and, Label: I-Degree
Token: graducation, Label: I-Degree
Token: in, Label: I-Degree
Token: Computer, Label: I-Degree
Token: Science, Label: I-Degree
Token: from, Label: O
Token: LUMS., Label: B-ORG
